## KGCN
#### This code is cloned from [this repository](https://github.com/zzaebok/KGCN-pytorch) which is an implementation of Knowledge GCN (KGCN) for MovieLens-1M datsaet. We craete KGAT and KGATSAGE models based on this model and train them in the KGAT.ipynb being provided in the same folder.  

In [ ]:
import pandas as pd
import numpy as np
import argparse
import random
from KGCNModel import KGCN
from baseline_data_loader import DataLoader
import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score

## Hyperparameters Settings

In [2]:
# prepare arguments (hyperparameters)
parser = argparse.ArgumentParser()

parser.add_argument('--dataset', type=str, default='movie', help='which dataset to use')
parser.add_argument('--aggregator', type=str, default='sum', help='which aggregator to use')
parser.add_argument('--n_epochs', type=int, default=20, help='the number of epochs')
parser.add_argument('--neighbor_sample_size', type=int, default=8, help='the number of neighbors to be sampled')
parser.add_argument('--dim', type=int, default=16, help='dimension of user and entity embeddings')
parser.add_argument('--n_iter', type=int, default=1, help='number of iterations when computing entity representation')
parser.add_argument('--batch_size', type=int, default=32, help='batch size')
parser.add_argument('--l2_weight', type=float, default=1e-4, help='weight of l2 regularization')
parser.add_argument('--lr', type=float, default=5e-4, help='learning rate')
parser.add_argument('--ratio', type=float, default=0.8, help='size of training dataset')

args = parser.parse_args(['--l2_weight', '1e-4', '--lr', '1e-3'])

In [3]:
# build dataset and knowledge graph
data_loader = DataLoader(args.dataset)
kg = data_loader.load_kg()
df_dataset = data_loader.load_dataset()
df_dataset

Construct knowledge graph ... Done
Build dataset dataframe ... Done


,userID,itemID,rating
0,0,742,5
1,0,433,3
2,0,580,3
3,0,2107,4
4,0,1432,5
...,...,...,...
664220,6039,671,1
664221,6039,674,5
664222,6039,370,5
664223,6039,676,4


In [22]:
# Dataset class
class KGCNDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        user_id = np.array(self.df.iloc[idx]['userID'])
        item_id = np.array(self.df.iloc[idx]['itemID'])
        rating = np.array(self.df.iloc[idx]['rating'], dtype=np.float32)
        return user_id, item_id, rating

In [23]:
# train test split
x_train, x_test = train_test_split(df_dataset, test_size=1 - args.ratio, shuffle=False, random_state=999)
train_dataset = KGCNDataset(x_train)
test_dataset = KGCNDataset(x_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=args.batch_size)

def RMSELoss(outouts, ratings):
    mse_loss = torch.nn.MSELoss()
    return torch.sqrt(mse_loss(outouts, ratings))

def train(net, optimizer, criterion, device, train_loader, test_loader):
    loss_list = []
    test_loss_list = []
    for epoch in range(args.n_epochs):
        running_loss = 0.0
        net.train()
        for i, (user_ids, item_ids, ratings) in enumerate(train_loader):
            user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
            optimizer.zero_grad()
            outputs = net(user_ids, item_ids)
            outputs = outputs.flatten()
            loss = criterion(outputs, ratings)
            loss.backward()
            
            optimizer.step()

            running_loss += loss.item()
    
        # print train loss per every epoch
        print('[Epoch {}]train_loss: '.format(epoch+1), running_loss / len(train_loader))
        loss_list.append(running_loss / len(train_loader))
        
        # evaluate per every epoch
        net.eval()
        with torch.no_grad():
            test_loss = 0
            #total_acc = 0
            for user_ids, item_ids, ratings in test_loader:
                user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
                outputs = net(user_ids, item_ids)
                outputs = torch.clip(outputs, 1, 5)
                outputs = outputs.flatten()
                test_loss += criterion(outputs, ratings).item()
            print('[Epoch {}]test_loss: '.format(epoch+1), test_loss / len(test_loader))
            test_loss_list.append(test_loss / len(test_loader))
    return loss_list, test_loss_list    

## Training KGCN Model

In [25]:
random_seed = 49
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
num_user, num_entity, num_relation = data_loader.get_num()
user_encoder, entity_encoder, relation_encoder = data_loader.get_encoders()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
kgcn = KGCN(num_user, num_entity, num_relation, kg, args, device).to(device)
optimizer = optim.Adam(kgcn.parameters(), lr=0.001, weight_decay=args.l2_weight)

kgcn_loss_list, kgcn_test_loss_list = train(kgcn, optimizer, RMSELoss, device, train_loader, test_loader)


[Epoch 1]train_loss:  1.2889741167244848
[Epoch 1]test_loss:  1.103806339254556
[Epoch 2]train_loss:  1.022659414998023
[Epoch 2]test_loss:  1.1015898673606745
[Epoch 3]train_loss:  1.0161481429823305
[Epoch 3]test_loss:  1.1021978243941402
[Epoch 4]train_loss:  1.0135049508624545
[Epoch 4]test_loss:  1.102139653411904
[Epoch 5]train_loss:  1.0110522584392265
[Epoch 5]test_loss:  1.1017192414385735
[Epoch 6]train_loss:  1.0095283275542597
[Epoch 6]test_loss:  1.1013022327009654
[Epoch 7]train_loss:  1.0079726064300647
[Epoch 7]test_loss:  1.100886196996263
[Epoch 8]train_loss:  1.0073470225642707
[Epoch 8]test_loss:  1.1009284213673403
[Epoch 9]train_loss:  1.0067181030870933
[Epoch 9]test_loss:  1.1004701336784743
[Epoch 10]train_loss:  1.007334402427535
[Epoch 10]test_loss:  1.100483882870805
[Epoch 11]train_loss:  1.0066233678465937
[Epoch 11]test_loss:  1.100279120258226
[Epoch 12]train_loss:  1.0061182705886276
[Epoch 12]test_loss:  1.1004825088344443
[Epoch 13]train_loss:  1.0055

In [26]:
torch.save(kgcn.state_dict(), '../checkpoints/KGCN.pth')